# Benchmark Plotting Explorer
Interactive plots for run convergence and empirical scaling exponent analysis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update(
    {
        "axes.titlesize": 14,
        "axes.titleweight": "normal",
        "figure.titlesize": 18,
        "figure.titleweight": "bold",
    }
)

RESULTS_DIR = Path("Results")

per_run_df = pd.read_csv(RESULTS_DIR / "per_run_times.csv")
per_run_df["is_warmup"] = per_run_df["is_warmup"].astype(bool)

case_df = pd.read_csv(RESULTS_DIR / "per_case_results.csv")

print(f"per_run_times : {len(per_run_df):,} rows")
print(f"per_case_results: {len(case_df):,} rows")
print(f"Algorithms : {sorted(per_run_df['algorithm'].unique())}")
print(f"Scenarios  : {sorted(per_run_df['scenario'].unique())}")
print(f"Sizes      : {sorted(per_run_df['target_size'].unique())}")

ALGORITHM_ORDER = ["CPython", "CPython JIT", "PyPy", "Cython"]
ALGORITHM_COLORS = {
    "CPython": "#1f3a5f",
    "CPython JIT": "#2a7f62",
    "PyPy": "#c26d1f",
    "Cython": "#7d2e68",
}
SCENARIO_LABELS = {
    "homologous_region": "Homologous region",
    "indel_disruption": "Indel disruption",
    "conserved_motif": "Conserved motif",
    "contained_fragment": "Contained fragment",
    "random_uniform": "Random uniform",
}

## Run Convergence
How runtime evolves across runs (warmup + timed). Warmup region is shaded.

In [ ]:
# ── Filters ────────────────────────────────────────────────────────────────
# Set to None to include all values, or a list to restrict.

CONV_FILTER_SCENARIOS  = ["random_uniform"]   # e.g. ['conserved_motif', 'contained_fragment', 'homologous_region', 'indel_disruption', 'random_uniform']
CONV_FILTER_SIZES      = None   # e.g. [100, 1000]
CONV_FILTER_ALGORITHMS = None   # e.g. ["CPython JIT", "PyPy"]

# ── Display options ─────────────────────────────────────────────────────────
CONV_FACET_BY            = "target_size"  # "target_size" or "scenario"
CONV_SHOW_INDIVIDUAL     = False          # True = thin line per case behind the median
CONV_RELATIVE            = True           # True = relative to stable median; False = raw ms

In [ ]:
df = per_run_df.copy()
if CONV_FILTER_SCENARIOS:
    df = df[df["scenario"].isin(CONV_FILTER_SCENARIOS)]
if CONV_FILTER_SIZES:
    df = df[df["target_size"].isin(CONV_FILTER_SIZES)]
if CONV_FILTER_ALGORITHMS:
    df = df[df["algorithm"].isin(CONV_FILTER_ALGORITHMS)]

real_df = df[~df["is_warmup"]]
ref_times = (
    real_df.groupby(["case_id", "algorithm"])["time_ms"]
    .median()
    .rename("ref_time_ms")
    .reset_index()
)
df = df.merge(ref_times, on=["case_id", "algorithm"], how="inner")
df = df[df["ref_time_ms"] > 0].copy()
df["relative_time"] = df["time_ms"] / df["ref_time_ms"]
y_col = "relative_time" if CONV_RELATIVE else "time_ms"
y_label = "Relative runtime [×]" if CONV_RELATIVE else "Runtime [ms]"

warmup_count = int(df[df["is_warmup"]]["run_index"].max()) + 1 if df["is_warmup"].any() else 0
total_runs = int(df["run_index"].max()) + 1
facet_values = sorted(df[CONV_FACET_BY].unique())
algorithms = [a for a in ALGORITHM_ORDER if CONV_FILTER_ALGORITHMS is None or a in CONV_FILTER_ALGORITHMS]

# Match plotting.py: max 2 columns, spare slot used for legend
n_cols = min(2, max(1, len(facet_values)))
n_rows = -(-len(facet_values) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.5 * n_rows), squeeze=False)
axes_flat = list(axes.ravel())

for ax, facet_val in zip(axes_flat, facet_values):
    facet_df = df[df[CONV_FACET_BY] == facet_val]
    for algorithm in algorithms:
        alg_df = facet_df[facet_df["algorithm"] == algorithm]
        if alg_df.empty:
            continue
        color = ALGORITHM_COLORS.get(algorithm, "gray")
        if CONV_SHOW_INDIVIDUAL:
            for _, case_df in alg_df.groupby("case_id"):
                case_df = case_df.sort_values("run_index")
                ax.plot(case_df["run_index"], case_df[y_col], color=color, alpha=0.15, linewidth=0.8)
        agg = alg_df.groupby("run_index")[y_col].median().reset_index().sort_values("run_index")
        ax.plot(agg["run_index"], agg[y_col], marker="o", linewidth=2, markersize=4, color=color, label=algorithm)

    if warmup_count > 0:
        ax.axvspan(-0.5, warmup_count - 0.5, alpha=0.10, color="#888888")
        ax.axvline(warmup_count - 0.5, color="#888888", linestyle=":", linewidth=1)
    if CONV_RELATIVE:
        ax.axhline(1.0, color="#555555", linestyle="--", linewidth=1)

    label = f"n = {facet_val:,}" if CONV_FACET_BY == "target_size" else SCENARIO_LABELS.get(facet_val, str(facet_val))
    ax.set_title(label)
    ax.set_xlabel("Run index")
    ax.set_ylabel(y_label)
    ax.set_xticks(range(total_runs))
    ax.grid(True, alpha=0.3)

handles, labels = axes_flat[0].get_legend_handles_labels()

# Put legend in the spare axes slot (if any), otherwise above the figure
unused = axes_flat[len(facet_values):]
if unused:
    legend_ax = unused[0]
    legend_ax.set_axis_off()
    legend_ax.legend(handles, labels, loc="lower right", frameon=False,
                     fontsize=14, handlelength=2.8, handletextpad=0.8,
                     labelspacing=1.0, borderaxespad=0.8, markerscale=1.4)
    for ax in unused[1:]:
        ax.set_visible(False)
else:
    fig.legend(handles, labels, loc="upper center", ncol=len(algorithms),
               frameon=False, fontsize=12, handlelength=2.8,
               handletextpad=0.8, labelspacing=0.8, markerscale=1.3)

title = "Runtime Convergence Across Runs"
if CONV_RELATIVE:
    title += " (relative to stable median)"
if warmup_count > 0:
    title += f"\nShaded = warmup ({warmup_count} run{'s' if warmup_count != 1 else ''})"
fig.suptitle(title)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## Empirical Scaling Exponent
OLS fit of $t \propto n^\alpha$ in log–log space per algorithm. Expected theoretical value is $\alpha = 2$ for $O(n^2)$.
Filter by scenario to check whether the exponent differs across input types.

In [ ]:
# ── Filters ────────────────────────────────────────────────────────────────
# Isolate specific scenarios to check if scaling behaviour differs between them.

EXP_FILTER_SCENARIOS  = ["contained_fragment"]   # e.g. ['conserved_motif', 'contained_fragment', 'homologous_region', 'indel_disruption', 'random_uniform']  — None = all combined
EXP_FILTER_ALGORITHMS = None   # e.g. ["CPython JIT", "PyPy"]

# ── Display options ─────────────────────────────────────────────────────────
EXP_FACET_BY_SCENARIO = True   # True = one subplot per scenario; False = all scenarios combined

In [ ]:
from matplotlib.lines import Line2D

def _fit_scaling_exponent(df: pd.DataFrame) -> pd.DataFrame:
    """OLS fit of log(t) ~ alpha * log(n) + const per algorithm. Returns alpha and 95% CI."""
    df = df.copy()
    df["log_n"] = np.log10(np.sqrt(df["sequence_a_length"] * df["sequence_b_length"]))
    df["log_t"] = np.log10(df["median_time_ms"].clip(lower=1e-9))

    records = []
    algorithms = [a for a in ALGORITHM_ORDER if EXP_FILTER_ALGORITHMS is None or a in EXP_FILTER_ALGORITHMS]
    for algorithm in algorithms:
        alg_df = df[df["algorithm"] == algorithm].dropna(subset=["log_n", "log_t"])
        if len(alg_df) < 3:
            continue
        log_n = alg_df["log_n"].values
        log_t = alg_df["log_t"].values
        A = np.column_stack([log_n, np.ones(len(log_n))])
        coeffs, *_ = np.linalg.lstsq(A, log_t, rcond=None)
        alpha = coeffs[0]
        residuals = log_t - A @ coeffs
        n_obs = len(log_n)
        ci = 0.0
        if n_obs > 2:
            s2 = np.sum(residuals**2) / (n_obs - 2)
            ATA_inv = np.linalg.inv(A.T @ A)
            ci = 1.96 * np.sqrt(s2 * ATA_inv[0, 0])
        records.append({"algorithm": algorithm, "alpha": alpha, "ci": ci})
    return pd.DataFrame(records)


def _draw_exponent_bars(ax, fit_df: pd.DataFrame, title: str) -> None:
    algorithms = fit_df["algorithm"].tolist()
    x = np.arange(len(algorithms))
    bar_width = 0.5
    for i, row in fit_df.iterrows():
        ax.bar(
            i,
            row["alpha"],
            width=bar_width,
            color=ALGORITHM_COLORS[row["algorithm"]],
            alpha=0.85,
        )
        ax.errorbar(i, row["alpha"], yerr=row["ci"], fmt="none", color="black", capsize=5, linewidth=1.5)
    ax.axhline(2.0, color="#555555", linestyle=":", linewidth=1.4)
    ax.set_xticks(x)
    ax.set_xticklabels(algorithms)
    ax.set_ylabel("Scaling exponent $\\alpha$  ($t \\propto n^\\alpha$)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis="y")


exp_df = case_df.copy()
if EXP_FILTER_SCENARIOS:
    exp_df = exp_df[exp_df["scenario"].isin(EXP_FILTER_SCENARIOS)]

if EXP_FACET_BY_SCENARIO:
    scenarios = sorted(exp_df["scenario"].unique())
    n_cols = min(3, len(scenarios))
    n_rows = -(-len(scenarios) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows), squeeze=False)
    axes_flat = list(axes.ravel())
    for ax, scenario in zip(axes_flat, scenarios):
        fit = _fit_scaling_exponent(exp_df[exp_df["scenario"] == scenario])
        _draw_exponent_bars(ax, fit, SCENARIO_LABELS.get(scenario, scenario))
    for ax in axes_flat[len(scenarios):]:
        ax.set_visible(False)
    fig.suptitle("Empirical Scaling Exponent per Scenario\n(OLS in log-log space, 95% CI error bars)")
else:
    fit = _fit_scaling_exponent(exp_df)
    fig, ax = plt.subplots(figsize=(8, 5))
    _draw_exponent_bars(ax, fit, "All scenarios combined")
    fig.suptitle("Empirical Scaling Exponent\n(OLS in log-log space, 95% CI error bars)")

handles, labels = [], []
for alg in ALGORITHM_ORDER:
    if EXP_FILTER_ALGORITHMS is None or alg in EXP_FILTER_ALGORITHMS:
        handles.append(plt.Rectangle((0, 0), 1, 1, color=ALGORITHM_COLORS[alg], alpha=0.85))
        labels.append(alg)
handles.append(Line2D([0], [0], color="#555555", linestyle=":", linewidth=1.4))
labels.append("Theoretical $\\alpha = 2$")
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
    fontsize=11,
    borderaxespad=0.0,
    handlelength=2.8,
    handletextpad=0.8,
    labelspacing=0.8,
    markerscale=1.2,
    ncol=1,
    )
fig.tight_layout(rect=(0, 0, 0.86, 0.93))
plt.show()

In [ ]:
save_path = RESULTS_DIR / "plots" / "scaling_exponent.pdf"
save_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", pad_inches=0.2)